In [7]:
# One Hot Encoding of Categoricxal Columns and Standard Scaling Of Numerical Column

In [8]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder,StandardScaler
import seaborn as sns
from sklearn.impute import SimpleImputer

In [9]:
df = sns.load_dataset('titanic')
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [10]:
df = df[['survived','pclass','sex','age','sibsp','parch','embarked']]

In [11]:
# Separate Feature and Label
X = df.drop('survived',axis=1)
y = df['survived']

In [13]:
# One Hot Encoding Categorical Features
cat_features = ['pclass','sex','sibsp','parch','embarked']
encoder = OneHotEncoder(sparse_output = False, handle_unknown = 'ignore')
# sparse_output = False -> Return Standard numpy array, not a scipy array(which is default)
# handle_unkown = 'ignore' -> If we see unknown values durin transform, ignore them

In [14]:
## 1. Fit the encoder of the categorical feature 
encoded_data = encoder.fit_transform(df[cat_features])

In [16]:
## 2. Get tge new feature names (Optional)
feature_names = encoder.get_feature_names_out(cat_features)

In [17]:
## 3. Create a new DataFrame with the encoded data and feature names
encoded_df = pd.DataFrame(encoded_data,columns = feature_names)
encoded_df.head()

,pclass_1,pclass_2,pclass_3,sex_female,sex_male,sibsp_0,sibsp_1,sibsp_2,sibsp_3,sibsp_4,...,parch_1,parch_2,parch_3,parch_4,parch_5,parch_6,embarked_C,embarked_Q,embarked_S,embarked_nan
0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [20]:
# Now Scale the numerical Age Column : Handle Missing Values in Age before Scaling
missing_values_age = X['age'].isna().any()
if missing_values_age:
    print('Handling Nulls...')
    imputer = SimpleImputer(strategy = "mean")
    imputer.fit(X[['age']])
    X['age'] = imputer.transform(X[['age']])[:,0]
print(X.age)

Handling Nulls...
0      22.000000
1      38.000000
2      26.000000
3      35.000000
4      35.000000
         ...    
886    27.000000
887    19.000000
888    29.699118
889    26.000000
890    32.000000
Name: age, Length: 891, dtype: float64


In [21]:
#  Standard Scale the age feature (after imputation)
scaler = StandardScaler()
scaler.fit(X[['age']])
X_scaled = X.copy() # Careate a copy to avoid modification original data
print(X_scaled)

     pclass     sex        age  sibsp  parch embarked
0         3    male  22.000000      1      0        S
1         1  female  38.000000      1      0        C
2         3  female  26.000000      0      0        S
3         1  female  35.000000      1      0        S
4         3    male  35.000000      0      0        S
..      ...     ...        ...    ...    ...      ...
886       2    male  27.000000      0      0        S
887       1  female  19.000000      0      0        S
888       3  female  29.699118      1      2        S
889       1    male  26.000000      0      0        C
890       3    male  32.000000      0      0        Q

[891 rows x 6 columns]


In [22]:
X_scaled.loc[:,'age'] = scaler.transform(X[['age']])
print(X_scaled)

     pclass     sex       age  sibsp  parch embarked
0         3    male -0.592481      1      0        S
1         1  female  0.638789      1      0        C
2         3  female -0.284663      0      0        S
3         1  female  0.407926      1      0        S
4         3    male  0.407926      0      0        S
..      ...     ...       ...    ...    ...      ...
886       2    male -0.207709      0      0        S
887       1  female -0.823344      0      0        S
888       3  female  0.000000      1      2        S
889       1    male -0.284663      0      0        C
890       3    male  0.177063      0      0        Q

[891 rows x 6 columns]


In [24]:
# Append the Scaled Age to the original encoded_df dataframe
df_merged = pd.concat([encoded_df,X_scaled['age']], axis=1)
df_merged.head()

,pclass_1,pclass_2,pclass_3,sex_female,sex_male,sibsp_0,sibsp_1,sibsp_2,sibsp_3,sibsp_4,...,parch_2,parch_3,parch_4,parch_5,parch_6,embarked_C,embarked_Q,embarked_S,embarked_nan,age
0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.592481
1,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.638789
2,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.284663
3,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.407926
4,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.407926
